In [1]:
import pandas as pd
import geopandas as gpd

In [2]:
address_gdf = gpd.read_parquet("../../data/processed/nsw_addressing/addresspoint_all.parquet")
zoning_gdf = gpd.read_parquet("../../data/processed/nsw_zoning/land_zoning.parquet")

In [3]:
print("Address rows:", len(address_gdf))
print("Zoning rows:", len(zoning_gdf))

print("Address CRS:", address_gdf.crs)
print("Zoning CRS:", zoning_gdf.crs)

print(address_gdf.columns)
print(zoning_gdf.columns)

Address rows: 4225113
Zoning rows: 112245
Address CRS: {"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "GeographicCRS", "name": "WGS 84", "datum_ensemble": {"name": "World Geodetic System 1984 ensemble", "members": [{"name": "World Geodetic System 1984 (Transit)"}, {"name": "World Geodetic System 1984 (G730)"}, {"name": "World Geodetic System 1984 (G873)"}, {"name": "World Geodetic System 1984 (G1150)"}, {"name": "World Geodetic System 1984 (G1674)"}, {"name": "World Geodetic System 1984 (G1762)"}, {"name": "World Geodetic System 1984 (G2139)"}, {"name": "World Geodetic System 1984 (G2296)"}], "ellipsoid": {"name": "WGS 84", "semi_major_axis": 6378137, "inverse_flattening": 298.257223563}, "accuracy": "2.0", "id": {"authority": "EPSG", "code": 6326}}, "coordinate_system": {"subtype": "ellipsoidal", "axis": [{"name": "Geodetic latitude", "abbreviation": "Lat", "direction": "north", "unit": "degree"}, {"name": "Geodetic longitude", "abbreviation": "Lon", "direct

In [5]:
zoning_keep = zoning_gdf[
    ["OBJECTID", "EPI_NAME", "LGA_NAME", "LAY_CLASS", "SYM_CODE", "PURPOSE", "EPI_TYPE", "geometry"]
].copy()

if address_gdf.crs != zoning_keep.crs:
    address_gdf = address_gdf.to_crs(zoning_keep.crs)

sample_address = address_gdf.sample(50000, random_state=42).copy()
print(len(sample_address))

50000


In [6]:
# spatial join
joined = gpd.sjoin(
    sample_address,
    zoning_keep,
    how="left",
    predicate="within"
)

In [7]:
print("Joined rows:", len(joined))
joined[["address", "SYM_CODE", "LAY_CLASS", "EPI_NAME", "LGA_NAME"]].head(10)

Joined rows: 78039


,address,SYM_CODE,LAY_CLASS,EPI_NAME,LGA_NAME
2262872,GRINTON AVENUE ASHMONT,RE1,Public Recreation,Wagga Wagga Local Environmental Plan 2010,WAGGA WAGGA
2262872,GRINTON AVENUE ASHMONT,RE1,Public Recreation,Wagga Wagga Local Environmental Plan 2010,WAGGA WAGGA
136524,58 VISTA AVENUE CATALINA,C2,Environmental Conservation,Eurobodalla Local Environmental Plan 2012,EUROBODALLA
136524,58 VISTA AVENUE CATALINA,C2,Environmental Conservation,Eurobodalla Local Environmental Plan 2012,EUROBODALLA
3400623,5/40 TRAIN STREET BROULEE,E1,Local Centre,Eurobodalla Local Environmental Plan 2012,EUROBODALLA
3400623,5/40 TRAIN STREET BROULEE,E1,Local Centre,Eurobodalla Local Environmental Plan 2012,EUROBODALLA
975625,13 BACH AVENUE EMERTON,R2,Low Density Residential,Blacktown Local Environmental Plan 2015,BLACKTOWN
1229084,12 GUILFOYLE PLACE CUDGEN,R2,Low Density Residential,Tweed Local Environmental Plan 2014,TWEED
1229084,12 GUILFOYLE PLACE CUDGEN,R2,Low Density Residential,Tweed Local Environmental Plan 2014,TWEED
3506896,14 SPURFIELD ROAD MCLEANS RIDGES,R5,Large Lot Residential,Lismore Local Environmental Plan 2012,LISMORE


In [8]:
joined["SYM_CODE"].isna().mean()

np.float64(0.002293724932405592)

In [9]:
# Distribution of zoning
joined["SYM_CODE"].value_counts(dropna=False).head(20)

SYM_CODE
R2     27877
R1     10865
R3      8227
R4      5830
MU1     4354
RU1     3767
E1      2283
C4      1945
RU5     1877
E4      1493
R5      1335
RU2     1138
E2      1008
E3       720
RE1      703
SP5      642
C3       594
RU4      495
SP2      420
DM       399
Name: count, dtype: int64

In [10]:
joined[
    ["address", "housenumber", "SYM_CODE", "LAY_CLASS", "EPI_NAME", "LGA_NAME"]
].sample(20, random_state=42)

,address,housenumber,SYM_CODE,LAY_CLASS,EPI_NAME,LGA_NAME
465930,156 GASKILL STREET CANOWINDRA,156,R1,General Residential,Cabonne Local Environmental Plan 2012,CABONNE
1686165,7 LYNDHURST STREET GLADESVILLE,7,R2,Low Density Residential,Ryde Local Environmental Plan 2014,RYDE
1237152,8 ANGLE VALE ROAD EDENSOR PARK,8,R2,Low Density Residential,Fairfield Local Environmental Plan 2013,FAIRFIELD
2209892,794 THE HORSLEY DRIVE SMITHFIELD,794,R2,Low Density Residential,Fairfield Local Environmental Plan 2013,FAIRFIELD
3738184,40/77 SHAFTESBURY ROAD BURWOOD,40/77,MU1,Mixed Use,Burwood Local Environmental Plan 2012,BURWOOD
3011895,72 GLENGARRIE ROAD GLENGARRIE,72,RU2,Rural Landscape,Tweed Local Environmental Plan 2014,TWEED
510698,62 GALLIPOLI AVENUE BLACKWALL,62,R1,General Residential,Central Coast Local Environmental Plan 2022,CENTRAL COAST
1943079,10 NORTON STREET LEICHHARDT,10,E1,Local Centre,Inner West Local Environmental Plan 2022,INNER WEST
2062942,27 GILL STREET BONALBO,27,RU5,Village,Kyogle Local Environmental Plan 2012,KYOGLE
1615203,25 CHAMBERLAIN AVENUE ROSE BAY,25,R2,Low Density Residential,Woollahra Local Environmental Plan 2014,WOOLLAHRA


In [11]:
joined["rid"].duplicated().mean()

np.float64(0.3592947116185497)

In [12]:
from pathlib import Path
Path("../../data/processed/site_features").mkdir(parents=True, exist_ok=True)

joined.to_parquet(
    "../../data/processed/site_features/address_with_zoning_sample.parquet",
    index=False
)